In [10]:
import numpy as np
import pickle
import os

In [11]:
edge_path = r"data/porto/network-porto/porto_edges_new_simplify.pkl"
node_path = r"data/porto/network-porto/porto_nodes_new.pkl"
output_path = r"examples/"
scaler_list = [
    "scaler_attr.pkl",
    "scaler_gps.pkl"
]
train_path = r"data/porto/train.npy"

In [12]:
with open(edge_path, 'rb') as f:
    edges = pickle.load(f)
    
with open(node_path, 'rb') as f:
    nodes = pickle.load(f)
    
train_data = np.load(train_path, allow_pickle=True)

In [13]:
linkids = []
dateinfo = []
inds = []
for d in train_data:
    linkids.append(np.asarray(d[1]))
    dateinfo.append(d[2:5])
    inds.append(d[0])
lens = np.asarray([len(k) for k in linkids], dtype=np.int16)    

def info(xs, date):
    infos = []
    length = 0
    for x in xs:
        info = edges[x]
        infot = []
        infot.append(info[1])
        infot.append(length)
        length += info[1]
        infot += list(date)
        try:
            infot += [
                nodes[info[2]][0],
                nodes[info[2]][1],
                nodes[info[3]][0],
                nodes[info[3]][1]
            ]
        except:
            print(info)
        infos.append(np.asarray(infot))
    return infos
con_links = np.concatenate([info(b, dateinfo[ind]) for ind, b in enumerate(linkids)], dtype='object')
print(con_links[:5])

[[480.51599999999996 0.0 5.0 194.0 233.0 -8.618880699999998
  41.155198799999994 -8.613308300000002 41.154164200000004]
 [48.62899999999999 480.51599999999996 5.0 194.0 233.0 -8.613308300000002
  41.154164200000004 -8.613250500000001 41.1537434]
 [110.236 529.145 5.0 194.0 233.0 -8.613250500000001 41.1537434
  -8.6119957 41.153597]
 [45.67 639.381 5.0 194.0 233.0 -8.6119957 41.153597 -8.611450999999999
  41.1535795]
 [117.186 685.0509999999999 5.0 194.0 233.0 -8.611450999999999 41.1535795
  -8.6100626 41.153502100000004]]


In [14]:
from sklearn.preprocessing import StandardScaler
from utils.util import StandardScaler2

In [15]:
print(con_links.shape)

(4399576, 9)


In [16]:
attr_data = con_links[:,0:2]
gps_data = con_links[:,5:9]

scaler_std = StandardScaler().fit(attr_data)
scaler_std2 = StandardScaler().fit(gps_data)

In [17]:
print("=== StandardScaler (cols 0:2) ===")
print("Mean :", scaler_std.mean_)
print("Scale:", scaler_std.scale_)

print("\n=== StandardScaler2 (cols 5:9) ===")
# assuming StandardScaler2 exposes similar attributes
print("Mean :", scaler_std2.mean_)
print("Scale:", scaler_std2.scale_)

=== StandardScaler (cols 0:2) ===
Mean : [ 107.49145989 3011.83751621]
Scale: [ 130.9217437  2749.28029108]

=== StandardScaler2 (cols 5:9) ===
Mean : [-8.62256805 41.15925935 -8.62265419 41.15931612]
Scale: [0.02513424 0.01239923 0.02518905 0.01246002]


In [18]:
with open(os.path.join(output_path,scaler_list[0]), 'wb') as f:
    pickle.dump(scaler_std, f)

with open(os.path.join(output_path,scaler_list[1]), 'wb') as f:
    pickle.dump(scaler_std2, f)